# Operational Reporting

Designing an operational report requires a completely different mindset than exploratory data analysis. You must prioritize speed, clarity, and binary outcomes: is this number passing or failing?

There are three core concepts to master:
1. **Core KPIs**: The primary metrics that sit at the top of the dashboard.
2. **RAG Status (Red, Amber, Green)**: Using conditional formatting to instantly flag operational failures.
3. **Actionable vs. Vanity Metrics**: Choosing numbers that actually drive human behavior.

Let's set up a Python sandbox to simulate a daily automated script that generates an Operational Report for a tech company's executive team.

In [ ]:
import pandas as pd
import numpy as np

# Simulate the last two days of operational data
np.random.seed(42)

data = {
    'Metric': [
        'Daily Active Users', 
        'New Signups', 
        'Server Uptime (%)', 
        'Customer Support Tickets', 
        'Total Page Views'
    ],
    'Target': [50000, 1500, 99.9, 200, 1000000],
    'Yesterday': [51000, 1600, 99.95, 180, 1200000],
    'Today': [48000, 1450, 98.20, 450, 1250000] # We had a server outage today!
}

df_ops = pd.DataFrame(data)

print("✅ Operational Data Loaded!")
display(df_ops)

✅ Operational Data Loaded!


,Metric,Target,Yesterday,Today
0,Daily Active Users,50000.0,51000.00,48000.0
1,New Signups,1500.0,1600.00,1450.0
2,Server Uptime (%),99.9,99.95,98.2
3,Customer Support Tickets,200.0,180.00,450.0
4,Total Page Views,1000000.0,1200000.00,1250000.0


# 1. Actionable vs. Vanity Metrics
If you put "Total Page Views" on an executive dashboard, you are wasting their time. Page views always go up. It feels good to look at, but if page views drop by 5%, what does the CEO actually *do*? Nothing. That is a **Vanity Metric**.

An **Actionable Metric** is tied to a specific business lever. "Server Uptime" is actionable. If it drops, the CEO immediately pages the Head of Engineering. 

In [9]:
# Remove the Vanity Metric from our operational report!
df_clean = df_ops[df_ops['Metric'] != 'Total Page Views'].copy()

print("--- Filtered out Vanity Metrics ---")
display(df_clean)

--- Filtered out Vanity Metrics ---


,Metric,Target,Yesterday,Today
0,Daily Active Users,50000.0,51000.00,48000.0
1,New Signups,1500.0,1600.00,1450.0
2,Server Uptime (%),99.9,99.95,98.2
3,Customer Support Tickets,200.0,180.00,450.0


# 2. Calculating the Delta (The "Compared To What?" Rule)
A number standing alone on a dashboard is useless. If an operational report says "Daily Active Users: 48,000", the immediate question is: *"Is that good or bad?"*

Every single prominent metric (large numeric KPI) on an operational report MUST have a comparison. Usually, this is **Day-over-Day (DoD)**, **Year-over-Year (YoY)**, or **Actual vs. Target**.

In [10]:
# Calculate the Day-over-Day Delta
df_clean['DoD_Change'] = df_clean['Today'] - df_clean['Yesterday']

# Calculate the % Variance from our daily Target
df_clean['Var_to_Target_%'] = ((df_clean['Today'] - df_clean['Target']) / df_clean['Target']) * 100

print("--- Added Context (Deltas and Variances) ---")
display(df_clean)

--- Added Context (Deltas and Variances) ---


,Metric,Target,Yesterday,Today,DoD_Change,Var_to_Target_%
0,Daily Active Users,50000.0,51000.00,48000.0,-3000.00,-4.000000
1,New Signups,1500.0,1600.00,1450.0,-150.00,-3.333333
2,Server Uptime (%),99.9,99.95,98.2,-1.75,-1.701702
3,Customer Support Tickets,200.0,180.00,450.0,270.00,125.000000


# 3. RAG Status (Red, Amber, Green)
Executives do not want to read spreadsheets. They want to glance at a screen and know within 2 seconds if the business is burning down. 

We use **RAG (Red, Amber, Green) thresholds** to visually flag issues. In Pandas, we can actually build a styled HTML report that simulates this exact dashboard behavior using the `.style` accessor!

In [11]:
# Define our RAG logic
def apply_rag_status(row):
    metric = row['Metric']
    variance = row['Var_to_Target_%']
    
    # Logic for metrics where HIGHER is better (Users, Signups, Uptime)
    if metric in ['Daily Active Users', 'New Signups', 'Server Uptime (%)']:
        if variance >= 0:
            color = '#d4edda' # Light Green
        elif variance >= -5:
            color = '#fff3cd' # Light Yellow (Amber)
        else:
            color = '#f8d7da' # Light Red
            
    # Logic for metrics where LOWER is better (Support Tickets)
    else: 
        if variance <= 0:
            color = '#d4edda' # Light Green
        elif variance <= 10:
            color = '#fff3cd' # Light Yellow (Amber)
        else:
            color = '#f8d7da' # Light Red
            
    # Apply the background color to the entire row
    return [f'background-color: {color}'] * len(row)

# Apply the styling to create our final Executive Report
styled_report = df_clean.style.apply(apply_rag_status, axis=1) \
                              .format({
                                  'Target': '{:,.2f}',
                                  'Yesterday': '{:,.2f}',
                                  'Today': '{:,.2f}',
                                  'DoD_Change': '{:+,.0f}', # Add + or - sign
                                  'Var_to_Target_%': '{:+.2f}%'
                              }) \
                              .set_caption("Morning Executive Briefing: System Outage Detected")

print("--- Final Automated Executive Report ---")
display(styled_report)

--- Final Automated Executive Report ---


,Metric,Target,Yesterday,Today,DoD_Change,Var_to_Target_%
0,Daily Active Users,"50,000.00","51,000.00","48,000.00","-3,000",-4.00%
1,New Signups,"1,500.00","1,600.00","1,450.00",-150,-3.33%
2,Server Uptime (%),99.90,99.95,98.20,-2,-1.70%
3,Customer Support Tickets,200.00,180.00,450.00,+270,+125.00%


*(Insight: Look at the output! Without reading a single number, the executive's eye is instantly drawn to the bright red rows. They immediately know that Server Uptime missed its target by -1.7%, causing Customer Support Tickets to explode by +125% over the target. This report took 1 second to read, and it instantly triggered a business action.)*

# 4. Refresh Cadence
The final piece of operational reporting is deciding **how often** the data updates. 
* **Real-Time (Streaming)**: Only for critical operations (e.g., fraudulent credit card blocks, live server monitoring). Extremely expensive to maintain.
* **Intra-day (Hourly/15-min)**: For highly volatile operations (e.g., e-commerce sales on Black Friday, call center queues).
* **Batch (Daily/Overnight)**: The standard for 90% of business reporting. The database processes everything at 2:00 AM, and the clean report is waiting in the CEO's inbox at 7:00 AM.

---

## Real-World Use Case or Analogy:
Think of Operational Reporting like the **Dashboard of your Car**:

* **Primary Display Metrics**: The Speedometer and the Gas Gauge. They are prominent, front-and-center, and tell you the current state of the vehicle at a glance.

* **The "Compared To What?" Rule**: The speed limit sign on the road next to you. Knowing you are going 65 MPH is useless unless you know the target is 45 MPH.

* **RAG Status (Alerts)**: The Check Engine light or the Low Tire Pressure light. They are completely invisible (white/grey) when things are fine, but they glow bright yellow or red the millisecond a threshold is breached, demanding your immediate action. 

* **Vanity Metrics**: The Odometer. It's cool to know your car has driven 150,000 miles in its lifetime, but looking at that number doesn't change how you press the gas pedal today. 

---